# Module 2: Sequential Chain (15 min)

Apply **Pattern 1** from the deck: break the single-agent ceiling by building a 3-stage pipeline where each agent has one focused job and passes its output to the next.

```
Decision Brief
      │
      ▼
 ┌────────────┐  callback_handler=None
 │ Researcher │  gathers data with tools
 └─────┬──────┘
       │ str(result)
       ▼
 ┌────────────┐  callback_handler=None
 │  Analyst   │  evaluates options A/B/C
 └─────┬──────┘
       │ str(result)
       ▼
 ┌─────────────┐  streams to participant
 │ Synthesizer │  writes executive memo
 └─────────────┘
```

**When to use this pattern:**
- Steps have a natural, fixed order
- Each stage depends on the previous
- You want simple, predictable, debuggable flow

**Prerequisites:** Complete Module 1 first — this module reuses its tools.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# ── Model configuration ──────────────────────────────────────────────────
# Option 1 — Claude Sonnet 4 (default):
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
# Option 2 — Claude Haiku 4.5 (faster):
#   model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")
# Option 3 — Amazon Nova Pro (AWS credits):
#   model = BedrockModel(model_id="amazon.nova-pro-v1:0")
# Option 4 — Amazon Nova Lite (cheapest):
#   model = BedrockModel(model_id="amazon.nova-lite-v1:0")
#
# Pass model= to Agent(...) to activate. Without it, Strands uses Claude Sonnet 4.

---

## Part 1 — Import Tools and Define Prompts

We reuse the three mock tools from Module 1. Each agent gets a **narrow system prompt** focused on exactly one role. The key insight: the docstring tells the model *when* to use a tool; the system prompt tells it *what job it has*.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "01-strands-foundations"))

from strands import Agent
from decision_brief_tools import get_company_data, get_market_benchmarks, get_competitor_data

print("Tools loaded: get_company_data | get_market_benchmarks | get_competitor_data")

In [ ]:
# ── System prompts — each agent has ONE job ───────────────────────────────
# Narrow, focused prompts are the key to quality in a sequential chain.
# Each agent reads only what it needs and returns only what the next needs.

RESEARCHER_PROMPT = '''You are a market research specialist.
Given a decision brief, gather relevant company data, market benchmarks,
and competitor intelligence using your tools.
Return structured findings — data only, no recommendations.'''

ANALYST_PROMPT = '''You are a business strategy analyst.
Given market research findings and a decision brief, analyze each option (A, B, C).
For each option return:
- Strengths and weaknesses
- Implementation complexity: Low / Medium / High (with one-sentence justification)
- Top 2 risks with specific mitigations
- Verdict: Proceed / Proceed with caution / Do not proceed
Return structured analysis only — no executive memo yet.'''

SYNTHESIZER_PROMPT = '''You are an executive communications specialist.
Given research findings, option analyses, and the original brief, write a leadership memo:

## Decision Memo: [Title]
**Recommendation**: [one sentence — which option and why]

### Options at a Glance
| | Option A | Option B | Option C |
|---|---|---|---|
| Complexity | | | |
| Risk level | | | |
| Verdict | | | |

### Top 3 Risks & Mitigations
### Success Metrics (3-5 KPIs with targets)
### Decision Required: owner · deadline · approvers needed

Under 400 words. Be direct.'''

print("Prompts defined: RESEARCHER | ANALYST | SYNTHESIZER")

---

## Part 2 — Create the Three Agents

`callback_handler=None` makes an agent **silent** — it runs its full loop and returns its result as a string, but does not stream anything to the notebook. Only the final Synthesizer streams, so the participant sees the memo being written in real time.

In [ ]:
# ── Create the three specialized agents ───────────────────────────────────
# callback_handler=None makes an agent silent — its output is not streamed
# to the notebook. Only the final Synthesizer streams to the participant.

import time

researcher = Agent(
    tools=[get_company_data, get_market_benchmarks, get_competitor_data],
    system_prompt=RESEARCHER_PROMPT,
    callback_handler=None,   # silent — passes its output to the next stage
)

analyst = Agent(
    system_prompt=ANALYST_PROMPT,
    callback_handler=None,   # silent — passes its output to the next stage
)

synthesizer = Agent(
    system_prompt=SYNTHESIZER_PROMPT,
    # no callback_handler=None — streams the final memo to the participant
)

print("Agents ready: researcher | analyst | synthesizer")

---

## Part 3 — Run the Pipeline

Each stage passes `str(result)` as input to the next. The output of one agent becomes the context for the next — that is the chain.

In [ ]:
DECISION_BRIEF = '''
DECISION BRIEF: NovaCart Premium Tier Launch

Company: NovaCart (2M active users, mid-size e-commerce)
Decision owners: VP Product + CFO approval required

Options to evaluate:
  Option A — Exclusive Premium: invite-only for top 10% of spenders, $19.99/mo
  Option B — Gradual Rollout: 5% A/B test pilot with kill-switch, $14.99/mo
  Option C — Full Launch: open to all users immediately, $12.99/mo + 30-day free trial

Success target: +15% CLV improvement within 6 months
Budget: $2M  |  Decision deadline: 2027-01-31
'''

# ── Step 1: Researcher ────────────────────────────────────────────────────
t0 = time.time()
print("Step 1/3 — Researcher gathering market data...")

research_result = researcher(
    f"Gather market data and competitive intelligence for this decision:\n{DECISION_BRIEF}"
)
research_text = str(research_result)

print(f"  Done in {time.time()-t0:.1f}s — {len(research_text)} chars of research findings")

In [ ]:
# ── Step 2: Analyst ───────────────────────────────────────────────────────
t1 = time.time()
print("Step 2/3 — Analyst evaluating all three options...")

analysis_result = analyst(
    f"Original brief:\n{DECISION_BRIEF}\n\nResearch findings:\n{research_text}"
)
analysis_text = str(analysis_result)

print(f"  Done in {time.time()-t1:.1f}s — {len(analysis_text)} chars of analysis")

In [ ]:
# ── Step 3: Synthesizer ───────────────────────────────────────────────────
print("\nStep 3/3 — Synthesizer writing the executive memo:")
print("─" * 60)

t2 = time.time()
memo_result = synthesizer(
    f"Original brief:\n{DECISION_BRIEF}\n\n"
    f"Research findings:\n{research_text}\n\n"
    f"Option analyses:\n{analysis_text}"
)

print(f"\n─" + "─" * 59)
print(f"Done in {time.time()-t2:.1f}s")

---

## Part 4 — Inspect the Pipeline

In [ ]:
# ── Inspect what each stage produced ─────────────────────────────────────
print("=== RESEARCH FINDINGS (first 400 chars) ===")
print(research_text[:400], "...")
print()
print("=== OPTION ANALYSIS (first 400 chars) ===")
print(analysis_text[:400], "...")
print()
print("=== PIPELINE METRICS ===")
print(f"Researcher — messages in context: {len(researcher.messages)}")
print(f"Analyst    — messages in context: {len(analyst.messages)}")
print(f"Synthesizer— messages in context: {len(synthesizer.messages)}")
print()
print("Notice: each agent has a SHORT context.")
print("The Researcher only sees the brief + tool results.")
print("The Analyst only sees brief + research.")
print("The Synthesizer only sees brief + research + analysis.")
print("No agent is asked to do everything — that is the pattern.")

In [ ]:
# ── Why this beats one agent doing everything ─────────────────────────────
# In Module 1 the single-agent ceiling had:
#   - 2 LLM cycles
#   - all 4 roles (researcher + 3 analysts + synthesizer) in ONE context
#
# Here each agent had:
#   - 1-2 LLM cycles
#   - ONE focused role
#   - A context that only grows with what it needs
#
# As the task scales (more options, more data, more criteria),
# the sequential chain stays predictable. The single agent does not.

r_summary = research_result.metrics.get_summary()
a_summary = analysis_result.metrics.get_summary()
m_summary = memo_result.metrics.get_summary()

print("Cycles per stage:")
print(f"  Researcher:  {r_summary.get('total_cycles', 'n/a')}")
print(f"  Analyst:     {a_summary.get('total_cycles', 'n/a')}")
print(f"  Synthesizer: {m_summary.get('total_cycles', 'n/a')}")

---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| Sequential Chain | Each stage output becomes next stage input |
| `callback_handler=None` | Silent intermediate agents — only the final agent streams |
| Focused context | Each agent only sees what it needs — no context bloat |
| Predictable flow | Fixed order, easy to debug step by step |

---

## What's Next

In **Module 3: Parallel / Fork-Join**, options A, B, and C are analyzed at the same time instead of in sequence. Same agents, DAG execution — latency drops sharply.

---

## Want a real multi-turn conversation?

```bash
cd samples/02-sequential-chain
pip install -r requirements.txt
python chat.py
```